# Minimal SLAM from Front-Facing (Worldcam) Video

This is a minimal, end-to-end example to run monocular SLAM on the front-facing worldcam stream.
Model choice: ORB-SLAM3 (monocular). It is a strong, stable baseline, widely used, and easy to run on a single camera.

Assumptions:
- You have a worldcam video ingested in DataJoint.
- You can build ORB-SLAM3 separately.
- You have a camera calibration (intrinsics) or can provide reasonable placeholders.


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=True)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
import os
from pathlib import Path
import subprocess

import datajoint as dj
import numpy as np
import pandas as pd
from element_interface.utils import find_full_path

from adamacs.pipeline import scan, event, model
from adamacs.paths import get_experiment_root_data_dir

def find_repo_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / 'dj_local_conf.json').exists():
            return parent
    return current

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('DataJoint:', dj.__version__)


## 1) Pick a scan and resolve the front-facing video

In this repo the front-facing camera is typically the worldcam (`mini2p1_worldcam`).


In [ ]:
SCAN_ID = 'scan9FU06R1Y'  # TODO: set target scan_id

scan_key = (scan.Scan & f"scan_id = '{SCAN_ID}'").fetch1('KEY')

VideoRec = getattr(model, 'VideoRecordingNew', None) or getattr(model, 'VideoRecording', None)
if VideoRec is None:
    raise ValueError('VideoRecording table not available in this environment')

rows = (VideoRec * VideoRec.File & scan_key).fetch('camera', 'file_path', as_dict=True)
root_dirs = get_experiment_root_data_dir()

def is_front_cam(name):
    lower = name.lower()
    return ('world' in lower) or ('front' in lower) or ('ego' in lower)

worldcam_path = None
worldcam_camera = None
for row in rows:
    camera = str(row['camera'])
    if not is_front_cam(camera):
        continue
    rel = Path(str(row['file_path']))
    full = rel if rel.is_absolute() else find_full_path(root_dirs, rel)
    worldcam_path = full
    worldcam_camera = camera
    break

if worldcam_path is None:
    raise ValueError('No front-facing worldcam video found for this scan')

print('Worldcam camera:', worldcam_camera)
print('Worldcam video:', worldcam_path)


## 2) Build timestamps for SLAM (from event.Event)

We use the worldcam frame events as timestamps. If multiple worldcam event types exist,
choose the one with the most frames.


In [ ]:
event_rows = (
    dj.U('event_type')
    .aggr(event.Event & scan_key, n='count(*)')
    .fetch(as_dict=True)
)

worldcam_event_types = []
for row in event_rows:
    et = row['event_type'].lower()
    if 'world' in et and 'frame' in et:
        worldcam_event_types.append(row)

if not worldcam_event_types:
    raise ValueError('No worldcam frame events found for this scan')

worldcam_event_types.sort(key=lambda r: r['n'], reverse=True)
best_event_type = worldcam_event_types[0]['event_type']
print('Using event_type:', best_event_type)

timestamps = (event.Event & scan_key & f"event_type = '{best_event_type}'").fetch(
    'event_start_time', order_by='event_start_time'
).astype(float)
print('timestamps:', len(timestamps))


## 3) Export frames and build a TUM-style dataset

ORB-SLAM3's `mono_tum` example expects a TUM-like layout:
- `rgb/` with numbered frames
- `rgb.txt` with `timestamp rgb/<filename>`


In [ ]:
out_dir = Path('output') / SCAN_ID / 'slam_worldcam'
rgb_dir = out_dir / 'rgb'
rgb_dir.mkdir(parents=True, exist_ok=True)

# Extract frames with ffmpeg
ffmpeg_cmd = [
    'ffmpeg', '-y', '-i', str(worldcam_path),
    '-qscale:v', '2', str(rgb_dir / '%06d.png')
]
print('FFmpeg command:')
print(' '.join(ffmpeg_cmd))

RUN_FFMPEG = False
if RUN_FFMPEG:
    subprocess.run(ffmpeg_cmd, check=True)

# Build rgb.txt (use min of frame count and timestamps)
frame_files = sorted(rgb_dir.glob('*.png'))
if frame_files:
    n = min(len(frame_files), len(timestamps))
    rgb_txt = out_dir / 'rgb.txt'
    with rgb_txt.open('w', encoding='utf-8') as handle:
        handle.write('# timestamp rgb
')
        for i in range(n):
            handle.write(f"{timestamps[i]:.6f} rgb/{frame_files[i].name}
")
    print('Wrote:', rgb_txt, 'frames:', n)
else:
    print('No frames found yet. Run ffmpeg to populate rgb/.')


## 4) Minimal ORB-SLAM3 config (fill in intrinsics)

ORB-SLAM3 needs camera intrinsics. Replace the placeholders with a calibration.


In [ ]:
import cv2

cap = cv2.VideoCapture(str(worldcam_path))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

# TODO: replace fx, fy, cx, cy with calibrated intrinsics
fx = 0.9 * width
fy = 0.9 * width
cx = width / 2.0
cy = height / 2.0

config_path = out_dir / 'orbslam3_worldcam.yaml'
config_lines = [
    '%YAML:1.0',
    '',
    'Camera.type: "PinHole"',
    f'Camera.fx: {fx:.6f}',
    f'Camera.fy: {fy:.6f}',
    f'Camera.cx: {cx:.6f}',
    f'Camera.cy: {cy:.6f}',
    '',
    'Camera.k1: 0.0',
    'Camera.k2: 0.0',
    'Camera.p1: 0.0',
    'Camera.p2: 0.0',
    'Camera.k3: 0.0',
    '',
    f'Camera.width: {width}',
    f'Camera.height: {height}',
    f'Camera.fps: {fps:.3f}',
    'Camera.RGB: 1',
    '',
    'ORBextractor.nFeatures: 1000',
    'ORBextractor.scaleFactor: 1.2',
    'ORBextractor.nLevels: 8',
    'ORBextractor.iniThFAST: 20',
    'ORBextractor.minThFAST: 7',
    '',
    'Viewer.KeyFrameSize: 0.05',
    'Viewer.KeyFrameLineWidth: 1.0',
    'Viewer.GraphLineWidth: 0.9',
    'Viewer.PointSize: 2.0',
    'Viewer.CameraSize: 0.08',
    'Viewer.CameraLineWidth: 3.0',
    'Viewer.ViewpointX: 0.0',
    'Viewer.ViewpointY: -0.7',
    'Viewer.ViewpointZ: -1.8',
    'Viewer.ViewpointF: 500.0',
]
config_path.write_text('
'.join(config_lines) + '
', encoding='utf-8')
print('Wrote:', config_path)


## 5) Run ORB-SLAM3 (monocular)

Build ORB-SLAM3 separately, then run the monocular TUM example.


In [ ]:
ORB_SLAM3_DIR = Path('/path/to/ORB_SLAM3')  # TODO
VOCAB_PATH = ORB_SLAM3_DIR / 'Vocabulary' / 'ORBvoc.txt'

cmd = [
    str(ORB_SLAM3_DIR / 'Examples' / 'Monocular' / 'mono_tum'),
    str(VOCAB_PATH),
    str(config_path),
    str(out_dir)
]
print('ORB-SLAM3 command:')
print(' '.join(cmd))

RUN_ORB = False
if RUN_ORB:
    subprocess.run(cmd, check=True)


## 6) Load the trajectory output

ORB-SLAM3 writes `CameraTrajectory.txt` in TUM format (t, tx, ty, tz, qx, qy, qz, qw).


In [ ]:
traj_path = out_dir / 'CameraTrajectory.txt'
if traj_path.exists():
    cols = ['t', 'tx', 'ty', 'tz', 'qx', 'qy', 'qz', 'qw']
    df = pd.read_csv(traj_path, delim_whitespace=True, comment='#', names=cols)
    print(df.head())
else:
    print('Trajectory not found yet:', traj_path)
